## Ai Software development Agent System

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.tools import tool
from pathlib import Path
import subprocess
import os

In [3]:
# ============================================================
# PROJECT WORKSPACE
# ============================================================

PROJECT_ROOT = Path("generated_projects").resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


In [4]:
def safe_path(path: str) -> Path:
    """
    Convert a relative project path into a safe absolute path.

    This prevents the agent from accessing files outside
    the generated project workspace.
    """

    target = (PROJECT_ROOT / path).resolve()

    if not str(target).startswith(str(PROJECT_ROOT)):
        raise ValueError("Access denied: path is outside project workspace.")

    return target

In [5]:
@tool
def create_file(path: str, content: str) -> str:
    """
    Create a new file inside the project workspace.

    Args:
        path: Relative path of the file.
        content: Content to write into the file.

    Example:
        create_file(
            "app/main.py",
            "print('Hello World')"
        )
    """

    try:

        file_path = safe_path(path)

        # Create parent directories
        file_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # Prevent accidental overwrite
        if file_path.exists():
            return f"File already exists: {path}"

        file_path.write_text(
            content,
            encoding="utf-8"
        )

        return f"File created successfully: {path}"

    except Exception as e:
        return f"Error creating file: {str(e)}"



In [6]:
path = 'test/main.py'
result = create_file.invoke({
    "path": path,
    "content": "print('Hello from main.py')"
})

print(result)

File already exists: test/main.py


In [7]:
path2 = 'test/abcd.py'
result = create_file.invoke({
    "path": path2,
    "content": "print('Another file testing')"
})

print(result)

File created successfully: test/abcd.py


In [8]:
file_path = safe_path(path)
file_path

WindowsPath('F:/Artificial_Intellgence/Agentic_Ai_Course/Agentic_Ai_Detail_Course/Code_Lectures/Phase_03_Langchain_Agent_Fundamental/Project/Ai-Software-Development-Agent-System/research/generated_projects/test/main.py')

In [9]:
# ============================================================
# 2. READ FILE
# ============================================================

@tool
def read_file(path: str) -> str:
    """
    Read the contents of a project file.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if not file_path.is_file():
            return f"Path is not a file: {path}"

        content = file_path.read_text(
            encoding="utf-8"
        )

        return content

    except Exception as e:
        return f"Error reading file: {str(e)}"


In [10]:
read = read_file.invoke({
    "path":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects\\test\\main.py"
})

read

'print("Hello from main.py")'

In [11]:
# ============================================================
# 3. UPDATE FILE
# ============================================================

@tool
def update_file(path: str, content: str) -> str:
    """
    Replace the entire contents of an existing file.

    The Developer Agent can use this tool when fixing code.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        file_path.write_text(
            content,
            encoding="utf-8"
        )

        return f"File updated successfully: {path}"

    except Exception as e:
        return f"Error updating file: {str(e)}"



In [12]:
update = update_file.invoke({
    "path":path,
    "content":"print('Agentic ai')"
})
update

'File updated successfully: test/main.py'

In [13]:
# ============================================================
# 4. DELETE FILE
# ============================================================

@tool
def delete_file(path: str) -> str:
    """
    Delete a file from the project workspace.
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if not file_path.is_file():
            return f"Path is not a file: {path}"

        file_path.unlink()

        return f"File deleted successfully: {path}"

    except Exception as e:
        return f"Error deleting file: {str(e)}"


In [14]:
delete = delete_file.invoke({
    "path":path
})

In [15]:
# ============================================================
# 5. LIST PROJECT FILES
# ============================================================

@tool
def list_files(directory: str = "") -> str:
    """
    List files and directories inside the project workspace.

    Example:
        list_files("")
        list_files("app")
    """

    try:

        directory_path = safe_path(directory)

        if not directory_path.exists():
            return f"Directory does not exist: {directory}"

        items = []

        for item in directory_path.rglob("*"):

            relative_path = item.relative_to(PROJECT_ROOT)

            if item.is_dir():
                items.append(f"[DIR]  {relative_path}")
            else:
                items.append(f"[FILE] {relative_path}")

        if not items:
            return "Directory is empty."

        return "\n".join(items)

    except Exception as e:
        return f"Error listing files: {str(e)}"


In [16]:
list_dir = list_files.invoke({
    "directory":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects"
})
list_dir

'[DIR]  .ruff_cache\n[DIR]  test\n[FILE] .ruff_cache\\.gitignore\n[DIR]  .ruff_cache\\0.16.4\n[FILE] .ruff_cache\\CACHEDIR.TAG\n[FILE] test\\abcd.py\n[FILE] .ruff_cache\\0.16.4\\4865624669287325240'

In [17]:
# ============================================================
# 6. RUN PYTHON FILE
# ============================================================

@tool
def run_python(path: str) -> str:
    """
    Run a Python file inside the project workspace.

    Example:
        run_python("app/main.py")
    """

    try:

        file_path = safe_path(path)

        if not file_path.exists():
            return f"File does not exist: {path}"

        if file_path.suffix != ".py":
            return "Only Python files can be executed."

        result = subprocess.run(
            ["python", str(file_path)],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=30
        )

        output = ""

        if result.stdout:
            output += f"STDOUT:\n{result.stdout}\n"

        if result.stderr:
            output += f"STDERR:\n{result.stderr}\n"

        output += f"\nReturn code: {result.returncode}"

        return output

    except subprocess.TimeoutExpired:
        return "Execution stopped: program exceeded 30 seconds."

    except Exception as e:
        return f"Error running Python file: {str(e)}"


In [18]:
run_file = run_python.invoke({
    "path":"F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects\\test\\main.py"
})
run_file

'File does not exist: F:\\Artificial_Intellgence\\Agentic_Ai_Course\\Agentic_Ai_Detail_Course\\Code_Lectures\\Phase_03_Langchain_Agent_Fundamental\\Project\\Ai-Software-Development-Agent-System\\research\\generated_projects\\test\\main.py'

In [19]:
# ============================================================
# 7. RUN PYTEST
# ============================================================

@tool
def run_tests() -> str:
    """
    Run pytest on the generated project.
    """

    try:

        result = subprocess.run(
            ["pytest", "-v"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=120
        )

        output = ""

        if result.stdout:
            output += result.stdout

        if result.stderr:
            output += "\nERRORS:\n"
            output += result.stderr

        output += f"\n\nReturn code: {result.returncode}"

        if result.returncode == 0:
            output += "\n\nTEST RESULT: PASSED"
        else:
            output += "\n\nTEST RESULT: FAILED"

        return output

    except subprocess.TimeoutExpired:
        return "Testing stopped: pytest exceeded 120 seconds."

    except Exception as e:
        return f"Error running tests: {str(e)}"


In [20]:
result = run_tests.invoke({})
print(result)

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- G:\InstallSoftWare\Anaconda_install\python.exe
cachedir: .pytest_cache
rootdir: F:\Artificial_Intellgence\Agentic_Ai_Course\Agentic_Ai_Detail_Course\Code_Lectures\Phase_03_Langchain_Agent_Fundamental\Project\Ai-Software-Development-Agent-System
configfile: pyproject.toml
plugins: anyio-4.14.2, langsmith-0.11.1
collecting ... collected 0 items

============================ no tests ran in 0.18s ============================


Return code: 5

TEST RESULT: FAILED


In [21]:
#  ============================================================
# 8. RUN RUFF
# ============================================================

@tool
def run_ruff() -> str:
    """
    Run Ruff code quality checks on the generated project.
    """

    try:

        result = subprocess.run(
            ["ruff", "check", "."],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=60
        )

        output = ""

        if result.stdout:
            output += result.stdout

        if result.stderr:
            output += "\nERRORS:\n"
            output += result.stderr

        output += f"\n\nReturn code: {result.returncode}"

        if result.returncode == 0:
            output += "\n\nCODE REVIEW: PASSED"
        else:
            output += "\n\nCODE REVIEW: ISSUES FOUND"

        return output

    except Exception as e:
        return f"Error running Ruff: {str(e)}"

In [22]:
result = run_ruff.invoke({})
print(result)

All checks passed!


Return code: 0

CODE REVIEW: PASSED


In [23]:
# ============================================================
# 9. GIT STATUS
# ============================================================

@tool
def git_status() -> str:
    """
    Show the current Git status of the generated project.
    """

    try:

        result = subprocess.run(
            ["git", "status", "--short"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return result.stdout or "Working tree is clean."

    except Exception as e:
        return f"Git error: {str(e)}"


In [24]:
result = git_status.invoke({})
result

' D test/main.py\n M ../trials.ipynb\n M ../../src/agents/agents.py\n?? test/abcd.py\n'

In [25]:
# ============================================================
# 10. GIT DIFF
# ============================================================

@tool
def git_diff() -> str:
    """
    Show changes made to the generated project.
    """

    try:

        result = subprocess.run(
            ["git", "diff"],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return result.stdout or "No changes detected."

    except Exception as e:
        return f"Git diff error: {str(e)}"


In [26]:
result = git_diff.invoke({})
result

'diff --git a/research/generated_projects/test/main.py b/research/generated_projects/test/main.py\ndeleted file mode 100644\nindex eedb4f3..0000000\n--- a/research/generated_projects/test/main.py\n+++ /dev/null\n@@ -1 +0,0 @@\n-print("Hello from main.py")\n\\ No newline at end of file\ndiff --git a/research/trials.ipynb b/research/trials.ipynb\nindex 7d77363..e5ed0d5 100644\n--- a/research/trials.ipynb\n+++ b/research/trials.ipynb\n@@ -85,7 +85,7 @@\n   },\n   {\n    "cell_type": "code",\n-   "execution_count": 6,\n+   "execution_count": 5,\n    "id": "ddd30733",\n    "metadata": {},\n    "outputs": [],\n@@ -134,7 +134,7 @@\n   },\n   {\n    "cell_type": "code",\n-   "execution_count": 7,\n+   "execution_count": 6,\n    "id": "7623aace",\n    "metadata": {},\n    "outputs": [\n@@ -158,7 +158,7 @@\n   },\n   {\n    "cell_type": "code",\n-   "execution_count": 10,\n+   "execution_count": 7,\n    "id": "4c90826b",\n    "metadata": {},\n    "outputs": [\n@@ -166,7 +166,7 @@\n      "name": 

In [27]:
# ============================================================
# 11. GIT COMMIT
# ============================================================

@tool
def git_commit(message: str) -> str:
    """
    Commit current project changes to Git.

    Example:
        git_commit("Add student API")
    """

    try:

        # Add changes
        subprocess.run(
            ["git", "add", "."],
            cwd=PROJECT_ROOT,
            check=True
        )

        # Commit
        result = subprocess.run(
            ["git", "commit", "-m", message],
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True
        )

        return (
            f"Git commit result:\n"
            f"{result.stdout}\n"
            f"{result.stderr}"
        )

    except Exception as e:
        return f"Git commit error: {str(e)}"

In [28]:
result = git_commit.invoke({
    "message":"Tools are created"
})
result

'Git commit result:\n[main 91c9bf7] Tools are created\n 2 files changed, 1 insertion(+), 1 deletion(-)\n create mode 100644 research/generated_projects/test/abcd.py\n delete mode 100644 research/generated_projects/test/main.py\n\n'

In [29]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv


In [30]:
load_dotenv()

True

In [31]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi". We need to respond appropriately. No policy issues. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.080077098, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.017807874, 'prompt_tokens_details': None, 'queue_time': 0.303502616, 'total_time': 0.097884972}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a03ee1-e44c-7700-bbfa-9edf0d54061a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [32]:
# ============================================================
# 1st Agent : Developer Agent
# ============================================================

def build_developer_agent():

    return create_agent(
        model=llm,
        tools=[
            create_file,
            read_file,
            update_file,
            delete_file,
            list_files,
        ],
    )


# ============================================================
# 2nd Agent : Code Execution Agent
# ============================================================

def build_execution_agent():

    return create_agent(
        model=llm,
        tools=[
            list_files,
            read_file,
            run_python,
        ],
    )


# ============================================================
# 3rd Agent : Testing Agent
# ============================================================

def build_testing_agent():

    return create_agent(
        model=llm,
        tools=[
            list_files,
            read_file,
            run_tests,
        ],
    )


# ============================================================
# 4th Agent : Debugging Agent
# ============================================================

def build_debugging_agent():

    return create_agent(
        model=llm,
        tools=[
            list_files,
            read_file,
            update_file,
            run_python,
            run_tests,
        ],
    )


# ============================================================
# 5th Agent : Code Review Agent
# ============================================================

def build_code_review_agent():

    return create_agent(
        model=llm,
        tools=[
            list_files,
            read_file,
            run_ruff,
        ],
    )


# ============================================================
# 6th Agent : Git Agent
# ============================================================

def build_git_agent():

    return create_agent(
        model=llm,
        tools=[
            git_status,
            git_diff,
            git_commit,
        ],
    )

In [33]:
manager_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an experienced software project manager.

Your job is to analyze the user's software requirements
and create a clear development plan.

Break the project into small, logical tasks.

Include:

- Project objective
- Functional requirements
- Non-functional requirements
- Required technologies
- Development tasks
- Testing requirements
- Expected project structure

Do NOT write implementation code.
Focus only on planning.
"""
    ),

    (
        "human",
        """
Create a development plan for the following software project:

{requirements}
"""
    ),
])


manager_chain = (
    manager_prompt
    | llm
    | StrOutputParser()
)

In [34]:

# ============================================================
# Architect Chain
# ============================================================

architect_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a senior software architect.

Design a complete architecture based on the project
requirements and development plan.

Include:

- Technology stack
- Project architecture
- Folder structure
- Components
- APIs if required
- Database design if required
- Dependencies
- Implementation sequence

Do NOT write the complete implementation.

Your output will be given to a Developer Agent.
"""
    ),

    (
        "human",
        """
PROJECT REQUIREMENTS:

{requirements}


DEVELOPMENT PLAN:

{plan}


Create an implementation-ready architecture.
"""
    ),
])


architect_chain = (
    architect_prompt
    | llm
    | StrOutputParser()
)

In [35]:
# ============================================================
# Documentation Chain
# ============================================================

documentation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a professional software documentation engineer.

Generate clear documentation for the completed software.

Include:

- Project overview
- Features
- Technology stack
- Installation
- Configuration
- Usage
- Project structure
- Testing
- Deployment
- Important notes
"""
    ),

    (
        "human",
        """
PROJECT REQUIREMENTS:

{requirements}


ARCHITECTURE:

{architecture}


TEST RESULTS:

{test_results}


CODE REVIEW:

{code_review}


Generate the final project documentation.
"""
    ),
])


documentation_chain = (
    documentation_prompt
    | llm
    | StrOutputParser()
)


In [36]:
# ============================================================
# Final Review / Critic Chain
# ============================================================

critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a strict senior software engineer and code reviewer.

Evaluate the completed software project.

Check:

- Requirement completion
- Architecture
- Implementation
- Testing
- Code quality
- Maintainability
- Security
- Documentation

Respond in the following format:

Score: X/10

Strengths:
- ...
- ...

Problems:
- ...
- ...

Recommended Improvements:
- ...
- ...

Final Verdict:
...
"""
    ),

    (
        "human",
        """
PROJECT REQUIREMENTS:

{requirements}


ARCHITECTURE:

{architecture}


TEST RESULTS:

{test_results}


CODE REVIEW:

{code_review}


DOCUMENTATION:

{documentation}

Evaluate the project strictly.
"""
    ),
])


critic_chain = (
    critic_prompt
    | llm
    | StrOutputParser()
)